# ColabDFT SPE — DFT single point

Runs a **DFT single point (SPE)** with [GPU4PySCF](https://github.com/pyscf/gpu4pyscf) on the input structure as it is. The structure is not optimized. No coding needed. UMA is not used, so no Hugging Face token is needed.

## How to use

1. In **Runtime → Change runtime type**, choose **T4 GPU**
2. In **1. Input form** below, set the output name, charge, spin multiplicity, functional and basis set
3. Choose **Runtime → Run all**
4. A file selection button appears in **2. File selection and checks**. Choose an xyz or a Gaussian gjf / com file. You can leave it running after that
5. Check the 3D view and energy in the cells below, and download the ZIP of results in the last cell

Only the structure is read from a gjf; the method, charge and multiplicity come from the form. If an xyz holds several structures, only the first is computed.

To run again, repeat steps 2–5 (the install is skipped). In the file selection, Cancel upload reuses the previous file.

## Output (contents of the ZIP)

| File | Contents |
|---|---|
| `<output name>_spe.xyz` | The computed structure. Energy and settings in the comment line |
| `<output name>_dft.csv` | DFT single point energy |
| `<output name>_log.txt` | Input settings, software versions, results |

## Notes

- DFT runs on GPU4PySCF (density fitting). The T4 is slow at double precision, so large molecules and def2-TZVPD take time. Without a GPU, DFT runs on the CPU with PySCF.
- Closed shells (multiplicity 1) use RKS, open shells UKS.
- Supported elements are H–Rn (Z = 1–86). Periodic systems (crystals, surfaces) are not supported.
- To optimize the structure with UMA before the single point, use [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt_en.ipynb).
- To see the electrostatic potential (ESP), use [ColabDFT_ESP](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_ESP_en.ipynb).

## License

- This notebook and the shared code (`colabuma`): [MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- Libraries installed at runtime are used under their own licenses. See [License in the README](https://github.com/RChibaChiba/ColabChem/blob/main/README.md#license)

## Citation

If you use this notebook in a publication, please cite:

- PySCF (Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020))
- GPU4PySCF (X. Wu *et al.*, arXiv:2404.09452 (2024))

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection happens here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, output, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

structure.check_multiplicity(multiplicity)
functional, basis = dft.parse_method(dft_functional, dft_basis)
job = dft.Job(structure.out_prefix(output_name), charge, multiplicity, functional, basis)
env.check_gpu()
upload_name, upload_data = structure.upload()
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install) { display-mode: "form" }
DEVICE = env.setup_dft()

In [ ]:
#@title 4. Prepare the structure { display-mode: "form" }
atoms, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity, uma=False)
structure.show_numbered(atoms)
ok(f"Structure ready: {atoms.get_chemical_formula()} ({len(atoms)} atoms).")

In [ ]:
#@title 5. DFT single point { display-mode: "form" }
dft_res = dft.run([("Input structure", atoms)], charge, multiplicity, functional, basis)
files, dft_log = dft.report(dft_res, None, geometry="input structure")
_p = dft_res["points"][0]
files["spe.xyz"] = structure.xyz_block(atoms.get_chemical_symbols(), atoms.positions,
                                       f"energy_Ha={_p['energy']:.10f} converged={_p['converged']} "
                                       f"method={functional}/{basis} charge={charge} multiplicity={multiplicity}")

In [ ]:
#@title 6. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabDFT SPE log", source_desc, job, [dft_log])
zip_path = output.save_zip(job.out_prefix, files)